# Food Delivery Time Prediction - Complete Analysis
## Objective
Predict food delivery times and classify deliveries as Fast/Delayed using machine learning models.

### Project Phases:
1. **Phase 1**: Data Collection & EDA
2. **Phase 2**: Predictive Modeling (Linear & Logistic Regression)
3. **Phase 3**: Reporting & Actionable Insights

## Phase 1: Data Collection and Exploratory Data Analysis (EDA)

### Step 1: Data Import and Preprocessing

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set styling
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load the dataset
df = pd.read_csv('/mnt/user-data/uploads/Food_Delivery_Time_Prediction.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst Few Rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())

### Handle Missing Values

In [ ]:
# Check for missing values
print("Missing Values:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values found")

# Check for duplicate rows
print(f"\nDuplicate Rows: {df.duplicated().sum()}")

# Display statistical summary
print("\nStatistical Summary:")
print(df.describe())

### Data Transformation

In [ ]:
# Make a copy for processing
df_processed = df.copy()

# Identify categorical and numerical columns
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical Columns:", categorical_cols)
print("\nNumerical Columns:", numerical_cols)

# Check unique values in categorical columns
print("\nUnique Values in Categorical Columns:")
for col in categorical_cols:
    print(f"{col}: {df_processed[col].unique()}")

In [ ]:
# Remove Order_ID and location columns (not needed for modeling)
df_processed = df_processed.drop(['Order_ID', 'Customer_Location', 'Restaurant_Location'], axis=1)

# One-Hot Encoding for categorical variables
categorical_to_encode = ['Weather_Conditions', 'Traffic_Conditions', 'Order_Priority', 'Order_Time', 'Vehicle_Type']

df_encoded = pd.get_dummies(df_processed, columns=categorical_to_encode, drop_first=True)

print("Shape after encoding:", df_encoded.shape)
print("\nEncoded columns:")
print(df_encoded.columns.tolist())

In [ ]:
# Normalize numerical features
from sklearn.preprocessing import StandardScaler

# Identify numerical columns (excluding target variable)
numerical_features = ['Distance', 'Delivery_Person_Experience', 'Restaurant_Rating', 
                     'Customer_Rating', 'Order_Cost', 'Tip_Amount']

scaler = StandardScaler()
df_encoded[numerical_features] = scaler.fit_transform(df_encoded[numerical_features])

print("Normalized numerical features")
print(df_encoded[numerical_features].describe())

### Step 2: Exploratory Data Analysis (EDA)

#### Descriptive Statistics

In [ ]:
# Descriptive statistics for target variable (Delivery_Time)
print("Delivery Time Statistics:")
print(f"Mean: {df['Delivery_Time'].mean():.2f} minutes")
print(f"Median: {df['Delivery_Time'].median():.2f} minutes")
print(f"Std Dev: {df['Delivery_Time'].std():.2f} minutes")
print(f"Min: {df['Delivery_Time'].min():.2f} minutes")
print(f"Max: {df['Delivery_Time'].max():.2f} minutes")
print(f"Variance: {df['Delivery_Time'].var():.2f}")

# Distribution of categorical variables
print("\n" + "="*50)
print("Distribution of Categorical Variables:")
print("="*50)
for col in ['Weather_Conditions', 'Traffic_Conditions', 'Order_Priority', 'Vehicle_Type']:
    print(f"\n{col}:")
    print(df[col].value_counts())

#### Visualization of Key Features

In [ ]:
# Distribution of Delivery Time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Delivery_Time'], bins=30, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Delivery Time (minutes)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Delivery Time')
axes[0].grid(axis='y', alpha=0.3)

axes[1].boxplot(df['Delivery_Time'], vert=True)
axes[1].set_ylabel('Delivery Time (minutes)')
axes[1].set_title('Boxplot of Delivery Time')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/01_delivery_time_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Distribution visualization saved!")

#### Outlier Detection

In [ ]:
# Detect outliers using IQR method
numerical_cols_original = ['Distance', 'Delivery_Person_Experience', 'Restaurant_Rating', 
                           'Customer_Rating', 'Delivery_Time', 'Order_Cost', 'Tip_Amount']

outliers_summary = {}
for col in numerical_cols_original:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outlier_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    outliers_summary[col] = outlier_count

print("Outliers Detected (IQR Method):")
for col, count in outliers_summary.items():
    print(f"{col}: {count} outliers")

# Visualize outliers
fig, axes = plt.subplots(2, 4, figsize=(16, 10))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols_original):
    axes[idx].boxplot(df[col])
    axes[idx].set_title(f'Boxplot: {col}')
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/02_outlier_detection.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nOutlier visualization saved!")

#### Correlation Analysis

In [ ]:
# Create correlation matrix for original numerical features
corr_cols = ['Distance', 'Delivery_Person_Experience', 'Restaurant_Rating', 
             'Customer_Rating', 'Delivery_Time', 'Order_Cost', 'Tip_Amount']

correlation_matrix = df[corr_cols].corr()

print("Correlation with Delivery_Time:")
print(correlation_matrix['Delivery_Time'].sort_values(ascending=False))

# Visualize correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=1)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/03_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nCorrelation visualization saved!")

### Step 3: Feature Engineering

In [ ]:
# Feature Engineering: Time-Based Features
df_encoded['is_rush_hour'] = ((df['Order_Time'] == 'Evening') | (df['Order_Time'] == 'Night')).astype(int)

# Feature Engineering: Create delivery speed category based on median delivery time
median_delivery_time = df['Delivery_Time'].median()
df_encoded['fast_delivery'] = (df['Delivery_Time'] <= median_delivery_time).astype(int)

print(f"Rush Hour Distribution: {df_encoded['is_rush_hour'].value_counts().to_dict()}")
print(f"\nFast Delivery Distribution: {df_encoded['fast_delivery'].value_counts().to_dict()}")

# Show engineered features impact
print(f"\nAverage Delivery Time during Rush Hour: {df[df_encoded['is_rush_hour'] == 1]['Delivery_Time'].mean():.2f} minutes")
print(f"Average Delivery Time during Non-Rush Hour: {df[df_encoded['is_rush_hour'] == 0]['Delivery_Time'].mean():.2f} minutes")

In [ ]:
# Visualize feature engineering impact
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rush Hour Analysis
rush_hour_data = [df[df_encoded['is_rush_hour'] == 0]['Delivery_Time'],
                   df[df_encoded['is_rush_hour'] == 1]['Delivery_Time']]
axes[0].boxplot(rush_hour_data, labels=['Non-Rush Hour', 'Rush Hour'])
axes[0].set_ylabel('Delivery Time (minutes)')
axes[0].set_title('Delivery Time: Rush Hour vs Non-Rush Hour')
axes[0].grid(axis='y', alpha=0.3)

# Traffic Conditions Impact
traffic_impact = df.groupby('Traffic_Conditions')['Delivery_Time'].agg(['mean', 'std'])
axes[1].bar(traffic_impact.index, traffic_impact['mean'], 
            yerr=traffic_impact['std'], capsize=5, color='coral', edgecolor='black')
axes[1].set_xlabel('Traffic Conditions')
axes[1].set_ylabel('Average Delivery Time (minutes)')
axes[1].set_title('Impact of Traffic Conditions on Delivery Time')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/04_feature_engineering_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Feature engineering analysis saved!")

## Phase 2: Predictive Modeling

### Step 4: Linear Regression Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Prepare features and target
X = df_encoded.drop('Delivery_Time', axis=1)
y = df_encoded['Delivery_Time']

# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")
print(f"Total features: {X_train.shape[1]}")

In [ ]:
# Train Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions
y_pred_train = lr_model.predict(X_train)
y_pred_test = lr_model.predict(X_test)

# Calculate evaluation metrics
lr_train_mse = mean_squared_error(y_train, y_pred_train)
lr_test_mse = mean_squared_error(y_test, y_pred_test)
lr_train_rmse = np.sqrt(lr_train_mse)
lr_test_rmse = np.sqrt(lr_test_mse)
lr_train_mae = mean_absolute_error(y_train, y_pred_train)
lr_test_mae = mean_absolute_error(y_test, y_pred_test)
lr_train_r2 = r2_score(y_train, y_pred_train)
lr_test_r2 = r2_score(y_test, y_pred_test)

print("="*60)
print("LINEAR REGRESSION MODEL - PERFORMANCE METRICS")
print("="*60)
print("\nTraining Set:")
print(f"  Mean Squared Error (MSE): {lr_train_mse:.4f}")
print(f"  Root Mean Squared Error (RMSE): {lr_train_rmse:.4f}")
print(f"  Mean Absolute Error (MAE): {lr_train_mae:.4f}")
print(f"  R² Score: {lr_train_r2:.4f}")

print("\nTest Set:")
print(f"  Mean Squared Error (MSE): {lr_test_mse:.4f}")
print(f"  Root Mean Squared Error (RMSE): {lr_test_rmse:.4f}")
print(f"  Mean Absolute Error (MAE): {lr_test_mae:.4f}")
print(f"  R² Score: {lr_test_r2:.4f}")
print("="*60)

In [ ]:
# Feature importance for Linear Regression
feature_importance_lr = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False).head(10)

print("\nTop 10 Most Important Features (Linear Regression):")
print(feature_importance_lr)

In [ ]:
# Visualize Linear Regression Results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Actual vs Predicted (Training)
axes[0, 0].scatter(y_train, y_pred_train, alpha=0.6, edgecolors='k')
axes[0, 0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Delivery Time')
axes[0, 0].set_ylabel('Predicted Delivery Time')
axes[0, 0].set_title(f'Training: Actual vs Predicted (R² = {lr_train_r2:.4f})')
axes[0, 0].grid(alpha=0.3)

# Actual vs Predicted (Testing)
axes[0, 1].scatter(y_test, y_pred_test, alpha=0.6, color='orange', edgecolors='k')
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 1].set_xlabel('Actual Delivery Time')
axes[0, 1].set_ylabel('Predicted Delivery Time')
axes[0, 1].set_title(f'Testing: Actual vs Predicted (R² = {lr_test_r2:.4f})')
axes[0, 1].grid(alpha=0.3)

# Residuals (Training)
residuals_train = y_train - y_pred_train
axes[1, 0].scatter(y_pred_train, residuals_train, alpha=0.6, edgecolors='k')
axes[1, 0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 0].set_xlabel('Predicted Delivery Time')
axes[1, 0].set_ylabel('Residuals')
axes[1, 0].set_title('Training: Residuals Plot')
axes[1, 0].grid(alpha=0.3)

# Residuals (Testing)
residuals_test = y_test - y_pred_test
axes[1, 1].scatter(y_pred_test, residuals_test, alpha=0.6, color='orange', edgecolors='k')
axes[1, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 1].set_xlabel('Predicted Delivery Time')
axes[1, 1].set_ylabel('Residuals')
axes[1, 1].set_title('Testing: Residuals Plot')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/05_linear_regression_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Linear regression visualization saved!")

In [ ]:
# Top features visualization
fig, ax = plt.subplots(figsize=(10, 6))
top_features = feature_importance_lr.head(10)
colors = ['green' if x > 0 else 'red' for x in top_features['Coefficient']]
ax.barh(top_features['Feature'], top_features['Coefficient'], color=colors, edgecolor='black')
ax.set_xlabel('Coefficient Value')
ax.set_title('Top 10 Most Important Features (Linear Regression)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/06_feature_importance_lr.png', dpi=300, bbox_inches='tight')
plt.show()

print("Feature importance visualization saved!")

### Step 5: Logistic Regression Model (Binary Classification)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc, roc_auc_score

# Target variable is already created: 'fast_delivery' (1 = Fast, 0 = Delayed)
y_binary = df_encoded['fast_delivery']
X_binary = df_encoded.drop(['Delivery_Time', 'fast_delivery'], axis=1)

# Train-Test Split
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_binary, y_binary, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train_bin.shape[0]}")
print(f"Testing set size: {X_test_bin.shape[0]}")
print(f"\nClass Distribution in Training Set:")
print(y_train_bin.value_counts())
print(f"\nClass Distribution in Testing Set:")
print(y_test_bin.value_counts())

In [ ]:
# Train Logistic Regression Model
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_bin, y_train_bin)

# Make predictions
y_pred_train_bin = log_model.predict(X_train_bin)
y_pred_test_bin = log_model.predict(X_test_bin)
y_pred_proba_train = log_model.predict_proba(X_train_bin)[:, 1]
y_pred_proba_test = log_model.predict_proba(X_test_bin)[:, 1]

# Calculate evaluation metrics
log_train_accuracy = accuracy_score(y_train_bin, y_pred_train_bin)
log_test_accuracy = accuracy_score(y_test_bin, y_pred_test_bin)
log_train_precision = precision_score(y_train_bin, y_pred_train_bin)
log_test_precision = precision_score(y_test_bin, y_pred_test_bin)
log_train_recall = recall_score(y_train_bin, y_pred_train_bin)
log_test_recall = recall_score(y_test_bin, y_pred_test_bin)
log_train_f1 = f1_score(y_train_bin, y_pred_train_bin)
log_test_f1 = f1_score(y_test_bin, y_pred_test_bin)
log_train_auc = roc_auc_score(y_train_bin, y_pred_proba_train)
log_test_auc = roc_auc_score(y_test_bin, y_pred_proba_test)

print("="*60)
print("LOGISTIC REGRESSION MODEL - PERFORMANCE METRICS")
print("="*60)
print("\nTraining Set:")
print(f"  Accuracy: {log_train_accuracy:.4f}")
print(f"  Precision: {log_train_precision:.4f}")
print(f"  Recall: {log_train_recall:.4f}")
print(f"  F1-Score: {log_train_f1:.4f}")
print(f"  AUC-ROC: {log_train_auc:.4f}")

print("\nTest Set:")
print(f"  Accuracy: {log_test_accuracy:.4f}")
print(f"  Precision: {log_test_precision:.4f}")
print(f"  Recall: {log_test_recall:.4f}")
print(f"  F1-Score: {log_test_f1:.4f}")
print(f"  AUC-ROC: {log_test_auc:.4f}")
print("="*60)

In [ ]:
# Confusion Matrix
cm_train = confusion_matrix(y_train_bin, y_pred_train_bin)
cm_test = confusion_matrix(y_test_bin, y_pred_test_bin)

print("\nConfusion Matrix - Training Set:")
print(cm_train)
print("\nConfusion Matrix - Test Set:")
print(cm_test)

In [ ]:
# Visualize Logistic Regression Results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Confusion Matrix - Training
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0], 
            xticklabels=['Delayed', 'Fast'], yticklabels=['Delayed', 'Fast'])
axes[0, 0].set_title('Training Set - Confusion Matrix')
axes[0, 0].set_ylabel('Actual')
axes[0, 0].set_xlabel('Predicted')

# Confusion Matrix - Testing
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Oranges', ax=axes[0, 1],
            xticklabels=['Delayed', 'Fast'], yticklabels=['Delayed', 'Fast'])
axes[0, 1].set_title('Test Set - Confusion Matrix')
axes[0, 1].set_ylabel('Actual')
axes[0, 1].set_xlabel('Predicted')

# ROC Curve - Training
fpr_train, tpr_train, _ = roc_curve(y_train_bin, y_pred_proba_train)
axes[1, 0].plot(fpr_train, tpr_train, label=f'Training (AUC = {log_train_auc:.4f})', lw=2)
axes[1, 0].plot([0, 1], [0, 1], 'r--', lw=2, label='Random Classifier')
axes[1, 0].set_xlabel('False Positive Rate')
axes[1, 0].set_ylabel('True Positive Rate')
axes[1, 0].set_title('ROC Curve - Training Set')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# ROC Curve - Testing
fpr_test, tpr_test, _ = roc_curve(y_test_bin, y_pred_proba_test)
axes[1, 1].plot(fpr_test, tpr_test, label=f'Testing (AUC = {log_test_auc:.4f})', lw=2, color='orange')
axes[1, 1].plot([0, 1], [0, 1], 'r--', lw=2, label='Random Classifier')
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate')
axes[1, 1].set_title('ROC Curve - Test Set')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/07_logistic_regression_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Logistic regression visualization saved!")

## Phase 3: Reporting and Insights

### Step 6: Model Evaluation and Comparison

In [ ]:
# Create comprehensive comparison table
comparison_data = {
    'Metric': ['Train Accuracy/R²', 'Test Accuracy/R²', 'Train MAE/Precision', 'Test MAE/Precision', 
               'Train RMSE/Recall', 'Test RMSE/Recall', 'Train MSE/F1', 'Test MSE/F1'],
    'Linear Regression': [
        f"{lr_train_r2:.4f}",
        f"{lr_test_r2:.4f}",
        f"{lr_train_mae:.4f}",
        f"{lr_test_mae:.4f}",
        f"{lr_train_rmse:.4f}",
        f"{lr_test_rmse:.4f}",
        f"{lr_train_mse:.4f}",
        f"{lr_test_mse:.4f}"
    ],
    'Logistic Regression': [
        f"{log_train_accuracy:.4f}",
        f"{log_test_accuracy:.4f}",
        f"{log_train_precision:.4f}",
        f"{log_test_precision:.4f}",
        f"{log_train_recall:.4f}",
        f"{log_test_recall:.4f}",
        f"{log_train_f1:.4f}",
        f"{log_test_f1:.4f}"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("MODEL COMPARISON - LINEAR REGRESSION vs LOGISTIC REGRESSION")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

In [ ]:
# Model comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Linear Regression Metrics
lr_metrics = ['R² (Train)', 'R² (Test)', 'MAE (Train)', 'MAE (Test)', 'RMSE (Train)', 'RMSE (Test)']
lr_values = [lr_train_r2, lr_test_r2, lr_train_mae/100, lr_test_mae/100, lr_train_rmse/100, lr_test_rmse/100]

axes[0].bar(lr_metrics, lr_values, color=['#1f77b4', '#1f77b4', '#ff7f0e', '#ff7f0e', '#2ca02c', '#2ca02c'],
            edgecolor='black', alpha=0.7)
axes[0].set_ylabel('Metric Value')
axes[0].set_title('Linear Regression - Performance Metrics')
axes[0].set_ylim([0, max(lr_values) * 1.1])
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Logistic Regression Metrics
log_metrics = ['Accuracy (Train)', 'Accuracy (Test)', 'Precision (Train)', 'Precision (Test)', 
               'Recall (Train)', 'Recall (Test)', 'F1 (Train)', 'F1 (Test)']
log_values = [log_train_accuracy, log_test_accuracy, log_train_precision, log_test_precision,
              log_train_recall, log_test_recall, log_train_f1, log_test_f1]

axes[1].bar(log_metrics, log_values, color=['#1f77b4', '#1f77b4', '#ff7f0e', '#ff7f0e', 
                                              '#2ca02c', '#2ca02c', '#d62728', '#d62728'],
            edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Metric Value')
axes[1].set_title('Logistic Regression - Performance Metrics')
axes[1].set_ylim([0, 1.1])
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/08_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Model comparison visualization saved!")

### Step 7: Actionable Insights and Recommendations

In [ ]:
# Actionable Insights
print("\n" + "="*80)
print("ACTIONABLE INSIGHTS & RECOMMENDATIONS")
print("="*80)

print("\n1. DELIVERY TIME ANALYSIS:")
print(f"   - Average delivery time: {df['Delivery_Time'].mean():.2f} minutes")
print(f"   - Median delivery time: {df['Delivery_Time'].median():.2f} minutes")
print(f"   - Standard deviation: {df['Delivery_Time'].std():.2f} minutes")
print(f"   - Range: {df['Delivery_Time'].min():.2f} to {df['Delivery_Time'].max():.2f} minutes")

print("\n2. TRAFFIC CONDITIONS IMPACT:")
traffic_stats = df.groupby('Traffic_Conditions')['Delivery_Time'].agg(['mean', 'count'])
print(traffic_stats.to_string())
print(f"   → High traffic increases avg delivery time by {(traffic_stats.loc['High', 'mean'] - traffic_stats.loc['Low', 'mean']):.2f} minutes")

print("\n3. WEATHER CONDITIONS IMPACT:")
weather_stats = df.groupby('Weather_Conditions')['Delivery_Time'].agg(['mean', 'count'])
print(weather_stats.to_string())

print("\n4. VEHICLE TYPE ANALYSIS:")
vehicle_stats = df.groupby('Vehicle_Type')['Delivery_Time'].agg(['mean', 'count'])
print(vehicle_stats.to_string())

print("\n5. RUSH HOUR IMPACT:")
rush_hour_stats = df.groupby(df_encoded['is_rush_hour'])['Delivery_Time'].agg(['mean', 'count'])
rush_hour_stats.index = ['Non-Rush Hour', 'Rush Hour']
print(rush_hour_stats.to_string())

print("\n" + "="*80)

In [ ]:
print("\n" + "="*80)
print("STRATEGIC RECOMMENDATIONS")
print("="*80)

print("\n✓ OPTIMIZING DELIVERY ROUTES:")
print("  - Implement real-time traffic monitoring systems")
print("  - Use predictive analytics to pre-route deliveries around high-traffic areas")
print(f"  - Focus on vehicles delivering >15 km (avg time: {df[df['Distance'] > 15]['Delivery_Time'].mean():.2f} min)")
print("  - Leverage distance as a strong predictor of delivery time")

print("\n✓ STAFFING OPTIMIZATION:")
print(f"  - Increase delivery personnel during evening/night hours ({df_encoded['is_rush_hour'].sum()} orders)")
print(f"  - High traffic periods need {((traffic_stats.loc['High', 'mean'] / traffic_stats.loc['Low', 'mean'] - 1) * 100):.1f}% more resources")
print("  - Prioritize experienced delivery personnel (experience shows positive correlation)")

print("\n✓ TRAINING IMPROVEMENTS:")
print(f"  - Delivery person experience has correlation: {correlation_matrix.loc['Delivery_Person_Experience', 'Delivery_Time']:.3f}")
print("  - Train delivery staff on:")
print("    • Efficient route planning")
print("    • Time management during peak hours")
print("    • Weather-specific delivery protocols")

print("\n✓ SYSTEM ENHANCEMENTS:")
print(f"  - High traffic conditions increase time by {((traffic_stats.loc['High', 'mean'] / traffic_stats.loc['Low', 'mean'] - 1) * 100):.1f}%")
print("  - Implement predictive algorithms for better ETA accuracy")
print("  - Monitor vehicle utilization by type and condition")
print(f"  - Fast deliveries: {(df_encoded['fast_delivery'].sum() / len(df_encoded) * 100):.1f}% of orders")

print("\n✓ CUSTOMER SATISFACTION:")
print(f"  - Customer rating correlation with delivery time: {correlation_matrix.loc['Customer_Rating', 'Delivery_Time']:.3f}")
print("  - Ensure faster deliveries to improve customer satisfaction")
print(f"  - High priority orders: {(df[df['Order_Priority'] == 'High'].shape[0] / len(df) * 100):.1f}% of total")

print("\n" + "="*80)

In [ ]:
# Create detailed insights visualization
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Weather Impact
ax1 = fig.add_subplot(gs[0, 0])
weather_data = df.groupby('Weather_Conditions')['Delivery_Time'].mean().sort_values()
ax1.barh(weather_data.index, weather_data.values, color='skyblue', edgecolor='black')
ax1.set_xlabel('Avg Delivery Time (min)')
ax1.set_title('Impact of Weather Conditions')
ax1.grid(axis='x', alpha=0.3)

# 2. Traffic Impact
ax2 = fig.add_subplot(gs[0, 1])
traffic_data = df.groupby('Traffic_Conditions')['Delivery_Time'].mean().sort_values()
ax2.barh(traffic_data.index, traffic_data.values, color='coral', edgecolor='black')
ax2.set_xlabel('Avg Delivery Time (min)')
ax2.set_title('Impact of Traffic Conditions')
ax2.grid(axis='x', alpha=0.3)

# 3. Vehicle Type Impact
ax3 = fig.add_subplot(gs[0, 2])
vehicle_data = df.groupby('Vehicle_Type')['Delivery_Time'].mean().sort_values()
ax3.barh(vehicle_data.index, vehicle_data.values, color='lightgreen', edgecolor='black')
ax3.set_xlabel('Avg Delivery Time (min)')
ax3.set_title('Impact of Vehicle Type')
ax3.grid(axis='x', alpha=0.3)

# 4. Order Priority Impact
ax4 = fig.add_subplot(gs[1, 0])
priority_data = df.groupby('Order_Priority')['Delivery_Time'].mean().sort_values()
ax4.barh(priority_data.index, priority_data.values, color='plum', edgecolor='black')
ax4.set_xlabel('Avg Delivery Time (min)')
ax4.set_title('Impact of Order Priority')
ax4.grid(axis='x', alpha=0.3)

# 5. Distance vs Delivery Time
ax5 = fig.add_subplot(gs[1, 1])
ax5.scatter(df['Distance'], df['Delivery_Time'], alpha=0.6, edgecolors='k', s=50)
ax5.set_xlabel('Distance (km)')
ax5.set_ylabel('Delivery Time (min)')
ax5.set_title(f'Distance vs Delivery Time (r={correlation_matrix.loc["Distance", "Delivery_Time"]:.3f})')
ax5.grid(alpha=0.3)

# 6. Experience vs Delivery Time
ax6 = fig.add_subplot(gs[1, 2])
ax6.scatter(df['Delivery_Person_Experience'], df['Delivery_Time'], alpha=0.6, edgecolors='k', s=50, color='orange')
ax6.set_xlabel('Delivery Person Experience')
ax6.set_ylabel('Delivery Time (min)')
ax6.set_title(f'Experience vs Delivery Time (r={correlation_matrix.loc["Delivery_Person_Experience", "Delivery_Time"]:.3f})')
ax6.grid(alpha=0.3)

# 7. Cost vs Delivery Time
ax7 = fig.add_subplot(gs[2, 0])
ax7.scatter(df['Order_Cost'], df['Delivery_Time'], alpha=0.6, edgecolors='k', s=50, color='green')
ax7.set_xlabel('Order Cost')
ax7.set_ylabel('Delivery Time (min)')
ax7.set_title(f'Order Cost vs Delivery Time (r={correlation_matrix.loc["Order_Cost", "Delivery_Time"]:.3f})')
ax7.grid(alpha=0.3)

# 8. Order Time Distribution
ax8 = fig.add_subplot(gs[2, 1])
order_time_data = df.groupby('Order_Time')['Delivery_Time'].mean().sort_values()
ax8.bar(order_time_data.index, order_time_data.values, color='steelblue', edgecolor='black')
ax8.set_ylabel('Avg Delivery Time (min)')
ax8.set_title('Impact of Order Time')
ax8.grid(axis='y', alpha=0.3)

# 9. Model Performance Comparison
ax9 = fig.add_subplot(gs[2, 2])
models = ['Linear\nRegression', 'Logistic\nRegression']
scores = [lr_test_r2, log_test_accuracy]
colors_bar = ['#1f77b4', '#ff7f0e']
ax9.bar(models, scores, color=colors_bar, edgecolor='black', alpha=0.7)
ax9.set_ylabel('Score')
ax9.set_title('Model Performance Comparison (Test Set)')
ax9.set_ylim([0, 1])
for i, v in enumerate(scores):
    ax9.text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')
ax9.grid(axis='y', alpha=0.3)

plt.savefig('/mnt/user-data/outputs/09_comprehensive_insights.png', dpi=300, bbox_inches='tight')
plt.show()

print("Comprehensive insights visualization saved!")

## Summary

In [ ]:
print("\n" + "#"*80)
print("#" + " "*78 + "#")
print("#" + " "*20 + "FOOD DELIVERY TIME PREDICTION - FINAL SUMMARY" + " "*14 + "#")
print("#" + " "*78 + "#")
print("#"*80)

print("\n📊 DATASET OVERVIEW:")
print(f"   Total Records: {len(df)}")
print(f"   Features: {df.shape[1]}")
print(f"   No Missing Values: ✓")

print("\n🎯 LINEAR REGRESSION MODEL:")
print(f"   Test R² Score: {lr_test_r2:.4f}")
print(f"   Test MAE: {lr_test_mae:.4f} minutes")
print(f"   Test RMSE: {lr_test_rmse:.4f} minutes")
print(f"   ✓ Good for predicting continuous delivery time values")

print("\n🎯 LOGISTIC REGRESSION MODEL:")
print(f"   Test Accuracy: {log_test_accuracy:.4f}")
print(f"   Test Precision: {log_test_precision:.4f}")
print(f"   Test Recall: {log_test_recall:.4f}")
print(f"   Test F1-Score: {log_test_f1:.4f}")
print(f"   Test AUC-ROC: {log_test_auc:.4f}")
print(f"   ✓ Good for classifying fast vs delayed deliveries")

print("\n🔑 KEY FINDINGS:")
print(f"   1. Distance is the strongest predictor (correlation: {correlation_matrix.loc['Distance', 'Delivery_Time']:.3f})")
print(f"   2. High traffic increases delivery time by ~{(traffic_stats.loc['High', 'mean'] - traffic_stats.loc['Low', 'mean']):.1f} minutes")
print(f"   3. Rush hours account for {df_encoded['is_rush_hour'].sum()} orders ({(df_encoded['is_rush_hour'].sum() / len(df_encoded) * 100):.1f}%)")
print(f"   4. Experience has negative correlation ({correlation_matrix.loc['Delivery_Person_Experience', 'Delivery_Time']:.3f})")
print(f"   5. {(df_encoded['fast_delivery'].sum() / len(df_encoded) * 100):.1f}% orders delivered faster than median")

print("\n💡 RECOMMENDATIONS:")
print("   ✓ Implement real-time traffic monitoring")
print("   ✓ Increase staffing during rush hours (evening/night)")
print("   ✓ Prioritize experienced delivery personnel")
print("   ✓ Use predictive models for accurate ETAs")
print("   ✓ Route optimization for long-distance deliveries")

print("\n" + "#"*80)
print("✅ Analysis Complete! All visualizations saved to outputs folder.")
print("#"*80)